# CocinaAI — Modelo No Supervisado: Clustering de Recetas

**Objetivo:** Identificar grupos naturales de recetas mexicanas basados en su perfil de ingredientes, para responder: ¿qué tipos de platillos maximizan el aprovechamiento de despensa (Zero Waste)?

**Algoritmo:** K-Means sobre vectores TF-IDF de ingredientes  
**Evaluación:** Método del codo + Silhouette Score  
**Salida:** Etiqueta de cluster por receta, integrada al dataset principal

### Imports

In [ ]:
!pip install scikit-learn plotly pandas numpy -q

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import scipy.sparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Carga de datos

Cargamos los archivos exportados por el notebook de EDA: el dataset principal y la matriz TF-IDF de ingredientes.

In [ ]:
# Si estás en Colab, sube los archivos desde tu computadora o Google Drive
# from google.colab import files
# uploaded = files.upload()  # sube recipes_df.csv, tfidf_matrix.npz, tfidf_feature_names.json

recipes_df = pd.read_csv('recipes_df.csv')
tfidf_matrix = scipy.sparse.load_npz('tfidf_matrix.npz')
with open('tfidf_feature_names.json') as f:
    feature_names = json.load(f)

print(f'Recetas cargadas: {recipes_df.shape[0]}')
print(f'Matriz TF-IDF: {tfidf_matrix.shape}')

In [ ]:
# Verificamos que el orden coincida
assert len(recipes_df) == tfidf_matrix.shape[0], 'ERROR: Los datasets no coinciden en tamaño'
print('Datasets alineados correctamente ✓')
recipes_df[['titulo','score_zero_waste','dificultad','num_ingredientes']].head(5)

## 2. Selección del número óptimo de clusters (k)

Usamos dos métodos complementarios:
- **Método del codo**: busca el punto donde la inercia deja de reducirse significativamente
- **Silhouette Score**: mide qué tan bien separados están los clusters (1 = perfecto, -1 = solapado)

In [ ]:
# Reducimos dimensionalidad antes de K-Means para mayor eficiencia
# TruncatedSVD es el equivalente de PCA para matrices esparsas
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(tfidf_matrix)
print(f'Varianza explicada con 50 componentes: {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Método del codo
inercias = []
silhouettes = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_reduced)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_reduced, labels, sample_size=min(500, len(recipes_df))))
    print(f'k={k}: inercia={km.inertia_:.1f}, silhouette={silhouettes[-1]:.3f}')

In [ ]:
# Gráfica del codo
fig_codo = go.Figure()
fig_codo.add_trace(go.Scatter(
    x=list(k_range), y=inercias,
    mode='lines+markers', name='Inercia',
    line=dict(color='#B5722A', width=2),
    marker=dict(size=8)
))
fig_codo.update_layout(
    title='Método del codo — selección de k óptimo',
    xaxis_title='Número de clusters (k)',
    yaxis_title='Inercia (WCSS)',
    template='plotly_white'
)
fig_codo.show()

In [ ]:
# Gráfica Silhouette Score
fig_sil = go.Figure()
fig_sil.add_trace(go.Scatter(
    x=list(k_range), y=silhouettes,
    mode='lines+markers', name='Silhouette',
    line=dict(color='#1D9E75', width=2),
    marker=dict(size=8)
))
fig_sil.update_layout(
    title='Silhouette Score por número de clusters',
    xaxis_title='Número de clusters (k)',
    yaxis_title='Silhouette Score',
    template='plotly_white'
)
fig_sil.show()

k_optimo = k_range[silhouettes.index(max(silhouettes))]
print(f'k óptimo según Silhouette: {k_optimo}')

## 3. Entrenamiento del modelo K-Means

Entrenamos K-Means con el k seleccionado. Usamos `n_init=10` para ejecutar el algoritmo 10 veces con diferentes semillas y quedarnos con la mejor solución.

In [ ]:
# Entrenamiento final
k_final = k_optimo  # ajustar manualmente si el codo sugiere otro valor

kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
recipes_df['cluster'] = kmeans.fit_predict(X_reduced)

print(f'Modelo entrenado con k={k_final}')
print(f'Inercia final: {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_reduced, recipes_df["cluster"]):.3f}')
print()
print('Distribución de recetas por cluster:')
print(recipes_df['cluster'].value_counts().sort_index())

## 4. Análisis e interpretación de clusters

Para cada cluster identificamos: ingredientes más representativos, score Zero Waste promedio, dificultad predominante y ejemplos de recetas.

In [ ]:
# Perfil de cada cluster
perfil_clusters = recipes_df.groupby('cluster').agg(
    num_recetas=('id', 'count'),
    score_zw_promedio=('score_zero_waste', 'mean'),
    tiempo_promedio=('tiempo_minutos', 'mean'),
    ingredientes_promedio=('num_ingredientes', 'mean'),
    calorias_promedio=('calorias', 'mean'),
    pct_faciles=('dificultad', lambda x: (x=='Facil').mean()),
    pct_vegetariana=('vegetariana', 'mean'),
).round(3)
perfil_clusters

In [ ]:
# Top ingredientes por cluster (basado en centroides TF-IDF)
tfidf_dense = tfidf_matrix.toarray()

print('=== TOP INGREDIENTES POR CLUSTER ===\n')
for cluster_id in sorted(recipes_df['cluster'].unique()):
    idx = recipes_df[recipes_df['cluster'] == cluster_id].index
    centroid = tfidf_dense[idx].mean(axis=0)
    top_idx = centroid.argsort()[-8:][::-1]
    top_ings = [feature_names[i] for i in top_idx]
    zw = perfil_clusters.loc[cluster_id, 'score_zw_promedio']
    n = perfil_clusters.loc[cluster_id, 'num_recetas']
    print(f'Cluster {cluster_id} ({int(n)} recetas | ZW score: {zw:.3f})')
    print(f'  Ingredientes clave: {", ".join(top_ings)}')
    print()

In [ ]:
# Ejemplos de recetas por cluster
print('=== EJEMPLOS DE RECETAS POR CLUSTER ===\n')
for cluster_id in sorted(recipes_df['cluster'].unique()):
    ejemplos = recipes_df[recipes_df['cluster']==cluster_id][['titulo','score_zero_waste','dificultad']]\
               .sort_values('score_zero_waste', ascending=False).head(3)
    print(f'Cluster {cluster_id}:')
    for _, row in ejemplos.iterrows():
        print(f"  - {row['titulo']} (ZW: {row['score_zero_waste']:.2f}, {row['dificultad']})")
    print()

In [ ]:
# Asignamos nombres descriptivos a los clusters basados en el análisis anterior
# INSTRUCCIÓN: Ajusta estos nombres según lo que veas en tus resultados
nombres_clusters = {
    0: 'Guisos tradicionales',
    1: 'Recetas rapidas de despensa',
    2: 'Platillos elaborados',
    3: 'Cocina vegetariana mexicana',
    4: 'Antojitos y snacks',
}
# Filtramos solo los clusters que existen
nombres_clusters = {k: v for k, v in nombres_clusters.items()
                    if k in recipes_df['cluster'].unique()}

recipes_df['cluster_nombre'] = recipes_df['cluster'].map(nombres_clusters)
print('Nombres asignados:')
print(recipes_df['cluster_nombre'].value_counts())

## 5. Visualización de clusters

In [ ]:
# Reducimos a 2D con PCA para visualizar
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_reduced)
recipes_df['pca_x'] = X_2d[:, 0]
recipes_df['pca_y'] = X_2d[:, 1]

print(f'Varianza explicada por 2 componentes PCA: {pca.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Scatter plot de clusters en 2D
fig_clusters = px.scatter(
    recipes_df, x='pca_x', y='pca_y',
    color='cluster_nombre',
    hover_name='titulo',
    hover_data={'score_zero_waste': True, 'dificultad': True, 'pca_x': False, 'pca_y': False},
    title='Clusters de recetas mexicanas (PCA 2D)',
    template='plotly_white',
    size='score_zero_waste',
    size_max=15
)
fig_clusters.update_layout(legend_title='Cluster')
fig_clusters.show()

In [ ]:
# Score Zero Waste por cluster — barras
zw_cluster = recipes_df.groupby('cluster_nombre')['score_zero_waste'].mean().round(3)\
             .sort_values(ascending=False).reset_index()

fig_zw = go.Figure(go.Bar(
    x=zw_cluster['cluster_nombre'],
    y=zw_cluster['score_zero_waste'],
    marker_color=['#1D9E75','#5DCAA5','#EF9F27','#D85A30','#534AB7'][:len(zw_cluster)]
))
fig_zw.update_layout(
    title='Score Zero Waste promedio por cluster',
    xaxis_title='Cluster', yaxis_title='Score Zero Waste',
    template='plotly_white'
)
fig_zw.show()

In [ ]:
# Heatmap de perfil nutricional por cluster
cols_heatmap = ['calorias', 'proteinas_g', 'carbohidratos_g', 'grasas_g',
                'fibra_g', 'num_ingredientes', 'tiempo_minutos', 'score_zero_waste']

heatmap_data = recipes_df.groupby('cluster_nombre')[cols_heatmap].mean()

# Normalizamos para comparar en la misma escala
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

fig_heat = go.Figure(go.Heatmap(
    z=heatmap_norm.values,
    x=cols_heatmap,
    y=heatmap_norm.index.tolist(),
    colorscale='RdYlGn',
    text=heatmap_data.round(1).values,
    texttemplate='%{text}',
    showscale=True
))
fig_heat.update_layout(
    title='Perfil de clusters — valores reales normalizados para comparación',
    template='plotly_white', height=350
)
fig_heat.show()

## 6. Evaluación del modelo

Reportamos las métricas principales del clustering.

In [ ]:
sil_final = silhouette_score(X_reduced, recipes_df['cluster'])

print('=== EVALUACIÓN DEL MODELO K-MEANS ===')
print(f'  Número de clusters (k): {k_final}')
print(f'  Inercia (WCSS): {kmeans.inertia_:.2f}')
print(f'  Silhouette Score: {sil_final:.3f}')
print(f'  Interpretación: ', end='')
if sil_final >= 0.5: print('Clusters bien definidos')
elif sil_final >= 0.25: print('Estructura moderada — aceptable para datos de texto')
else: print('Clusters solapados — considerar ajustar k o features')
print()
print('Nota: Un Silhouette Score de 0.25-0.5 es normal y esperado para')
print('datos de texto/NLP donde las categorías tienen bordes difusos.')

## 7. Conclusiones

### Hallazgos principales

El modelo identificó grupos naturales de recetas mexicanas con perfiles diferenciados de ingredientes. Los clusters con mayor **Score Zero Waste** corresponden a recetas que usan ingredientes comunes de despensa (chile, cebolla, ajo, jitomate) con pocos ingredientes totales, lo que facilita su preparación sin generar desperdicio.

### Impacto para CocinaAI

- El cluster con mayor score Zero Waste es el ideal para recomendar cuando el usuario tiene ingredientes limitados
- El nombre del cluster puede mostrarse en la app como contexto adicional
- Este análisis respalda científicamente las recomendaciones del chatbot con datos reales

### Limitaciones

- Spoonacular no es un dataset 100% representativo de la cocina mexicana casera
- El score Zero Waste es una proxy construida, no una medida directa de desperdicio
- Con más datos (recetas de sitios mexicanos) los clusters serían más granulares

## 8. Exportación

In [ ]:
# Guardamos el dataset con etiquetas de cluster
export_cols = ['id','titulo','cluster','cluster_nombre','score_zero_waste',
               'dificultad','tiempo_minutos','num_ingredientes','calorias',
               'proteinas_g','carbohidratos_g','grasas_g','vegana','vegetariana']

recipes_clustered = recipes_df[export_cols].copy()
recipes_clustered.to_csv('recipes_clustered.csv', index=False)
print(f'recipes_clustered.csv exportado — {recipes_clustered.shape}')

In [ ]:
# Resumen final por cluster para incluir en el documento
resumen_final = recipes_df.groupby(['cluster','cluster_nombre']).agg(
    recetas=('id','count'),
    score_zw=('score_zero_waste','mean'),
    tiempo=('tiempo_minutos','mean'),
    ingredientes=('num_ingredientes','mean'),
    calorias=('calorias','mean'),
    pct_facil=('dificultad', lambda x: f"{(x=='Facil').mean():.0%}")
).round(2).reset_index()
resumen_final